# Loan Default Prediction

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
#from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier
from imblearn.over_sampling import ADASYN
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data_descriptions = pd.read_csv('data_descriptions.csv')
pd.set_option('display.max_colwidth', None)
data_descriptions

## Load the data

In [ ]:
train_df = pd.read_csv("train.csv")
#test_df = pd.read_csv("test.csv")
print(train_df.shape)
#print(test_df.shape)

In [ ]:
print(train_df.head())

In [ ]:
print(test_df.head())

## Data preprocessing and visualization 

### Check for missing values(none in both train_df and test_df)

In [ ]:
print("Missing values:")
print(train_df.isnull().sum())
print(test_df.isnull().sum())

### Identify categorical and numerical features


In [ ]:
categorical_features = ['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']
numerical_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

### Default rate vs categorical features

In [ ]:
# Set up the matplotlib figure and axes
fig, axes = plt.subplots(nrows=1, ncols=len(categorical_features), figsize=(15, 5), sharey=True)

# Loop through the categorical features and create a bar plot for each
for ax, feature in zip(axes, categorical_features):
    # Calculate the default rates
    default_rates = train_df.groupby(feature)['Default'].mean()
    
    # Plot
    default_rates.plot(kind='bar', color='skyblue', ax=ax)
    #ax.set_title(f'Default Rates by {feature}')
    ax.set_xlabel(f'{feature}')
    ax.set_ylabel('Default Rate')
    ax.tick_params(axis='x', labelrotation=45)

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()

### Encode binary categorical features

In [ ]:
binary_features = ['HasMortgage', 'HasDependents', 'HasCoSigner']

for feature in binary_features:
    train_df[feature] = train_df[feature].map({'Yes': 1, 'No': 0})
    #test_df[feature] = test_df[feature].map({'Yes': 1, 'No': 0})

### Encode non-binary categorical features

In [ ]:
# Education levels are ordinal.
education_level = {
    "PhD": 3,
    "Master's": 2,
    "Bachelor's": 1,
    "High School": 0
}
train_df['Education'] = train_df['Education'].map(education_level) 

In [ ]:
'''
education_level = {
    "PhD": 0,
    "Master's": 1,
    "Bachelor's": 2,
    "High School": 3
}
'''

In [ ]:
One_hot_features = ['EmploymentType','MaritalStatus','LoanPurpose']
train_df = pd.get_dummies(train_df, columns=One_hot_features)

### Handling outliers
Boosting algorithms such as XGBoost are extremely sensitive to outliers. Thus, outliers should be removed before the data are trained.

In [ ]:
def treat_outliers(df,feature):
    '''
    feature: str
    df: data frame
    '''
    Q1 = np.nanquantile(df[feature], 0.25) 
    Q3 = np.nanquantile(df[feature], 0.75) 
    IQR = Q3 - Q1   # IQR Range
    Lower_Whisker = Q1 - 1.5*IQR  
    Upper_Whisker = Q3 + 1.5*IQR  
    df[feature] = np.clip(df[feature], Lower_Whisker, Upper_Whisker) # all the values samller than Lower_Whisker will be assigned value of Lower_whisker 
                                                            # and all the values above upper_whishker will be assigned value of upper_Whisker 
    return df

for f in numerical_features:
    train_df = treat_outliers(train_df,f)

In [ ]:
train_df.head()

In [ ]:
X = train_df.drop(['Default', 'LoanID'], axis=1)
y = train_df['Default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14,stratify=y)

### Class imbalance
In loan default prediction problem, one important issue is the class imbalance, as shown in the following histogram. In this project, we use ADASYN for the oversampling on the training data.

In [ ]:
# Districbution of loan default
plt.figure(figsize=(6, 4))
sns.countplot(x='Default', data=train_df)
plt.title('Distribution of Loan Default')
plt.xlabel('Default')
plt.ylabel('Count')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of loan default
plt.figure(figsize=(6, 4))
sns.countplot(x='Default', data=train_df)

# Add count labels on the bars
def show_counts(ax, total=len(train_df)):
    for p in ax.patches:
        count = p.get_height()
        x = p.get_x() + p.get_width() / 2
        y = p.get_height()
        percentage = '{:.1f}%'.format(100 * count / total)
        ax.annotate(f'{count}\n({percentage})', (x, y), ha='center', va='bottom')

show_counts(plt.gca())

plt.title('Distribution of Loan Default')
plt.xlabel('Default')
plt.ylabel('Count')
plt.show()

In [ ]:
print(X_train.dtypes)

In [ ]:
# SMOTE method

from imblearn.over_sampling import SMOTE
# Resampling the minority class. The strategy can be changed as required.
sm = SMOTE(sampling_strategy='minority', random_state=42)
# Fit the model to generate the data.
oversampled_X, oversampled_Y = sm.fit_resample(X_train, y_train)
oversampled = pd.concat([pd.DataFrame(oversampled_Y), pd.DataFrame(oversampled_X)], axis=1)
oversampled['Default'].value_counts()

In [ ]:
# ADASYN method

from imblearn.over_sampling import ADASYN
# Resampling the minority class. The strategy can be changed as required.
adasyn = ADASYN(sampling_strategy='minority', random_state=42)
# Fit the model to generate the data.
oversampled_X, oversampled_Y = adasyn.fit_resample(X_train, y_train)
oversampled = pd.concat([pd.DataFrame(oversampled_Y), pd.DataFrame(oversampled_X)], axis=1)
oversampled['Default'].value_counts()

In [ ]:
print(X_train_ada.head().to_string(index=False))

## XGBoost training

In [ ]:
param_grid = {
    'max_depth': [3,4,5],
    'min_child_weight': [1,2,3],
    'subsample': [0.5,0,6,0.7],
    'colsample_bytree': [0.5,0.6,0.7],
    'eta': [0.003,0.05,0.1],
    'n_estimators': [200,400,600],
    'gamma': [0,0.05,0.1],
    'lambda': [1,1.5,1.7],
    'alpha': [0.03,0.05,0.08]
}

# Calculate the ratio of negative to positive samples
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
# XGBoost model function
xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, scale_pos_weight=scale_pos_weight,eval_metric="auc")

grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='roc_auc', cv=5, verbose=1)
grid_search.fit(X_train, y_train)

#xgb_model.fit(X_train, y_train,verbose=True)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate the ROC AUC score
roc_auc = roc_auc_score(y_test, y_pred_proba)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC Score:", roc_auc)
print("Best hyperpara: ",grid_search.best_params_)